In [10]:
import numpy as np
import pandas as pd

WEEK 3 - 
1. Split data into X (features) and y (target)
2. Split into Train and Test sets
3. Apply SMOTE on training data only
4. Train 3 models: Logistic Regression, Random Forest, XGBoost
5. Compare them

In [12]:
df = pd.read_csv('loan_data_cleaned.csv')

print(df.shape)
print(df.columns.tolist())

(149986, 11)
['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


Loads the cleaned file we saved at the end of Week 2 (no need to repeat cleaning)
Confirms we have 149,986 rows and 11 columns, no Unnamed: 0

STEP1 - SPLIT INTO X(FEATURES) & Y(TARGET)

Think of your dataframe as two parts:

X = all the columns we use to PREDICT (the "clues")
y = the column we want to PREDICT (the "answer") → SeriousDlqin2yrs

It's like a practice exam: X = the questions, y = the answer key.

In [14]:
X = df.drop(columns=["SeriousDlqin2yrs"])   #take everything EXCEPT the target column → these are our features
y = df["SeriousDlqin2yrs"]                  #take ONLY the target column → this is what we predict
print('X shape:',X.shape)  
print('y shape:',y.shape)  

X shape: (149986, 10)
y shape: (149986,)


STEP 2 - TRAIN/TEST SPLIT

Concept first:
We can't train and test the model on the same data — that's like giving a student the exam questions AND the answer key beforehand, then being surprised they score 100%.
So we split:

Train set (e.g. 80%) → model learns patterns from this
Test set (e.g. 20%) → model has NEVER seen this — we use it to check if the model actually learned, or just memorized

Why We Need stratify
Remember our class imbalance — 93.3% vs 6.7%?
If we split randomly, by chance the test set might end up with even FEWER defaulters than 6.7% — making evaluation unreliable.

stratify=y tells Python: "Keep the same 93.3%/6.7% ratio in BOTH train and test sets."

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,       #20% goes to test, 80% to train
    random_state = 42,     #makes split reproducible - same split every time you run it
    stratify = y           #keep the 93.3%/6.7% ratio in both sets
)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (119988, 10)
X_test shape: (29998, 10)
y_train shape: (119988,)
y_test shape: (29998,)


STEP 3 - QUICK VERIFICATION

In [16]:
print('Train set class distribution')
print(y_train.value_counts(normalize = True) *100)

print('Test set class distribution')
print(y_test.value_counts(normalize = True) *100)

Train set class distribution
SeriousDlqin2yrs
0    93.315998
1     6.684002
Name: proportion, dtype: float64
Test set class distribution
SeriousDlqin2yrs
0    93.316221
1     6.683779
Name: proportion, dtype: float64


Both should show roughly 93.3% / 6.7% — same as the original data
This confirms stratify=y worked correctly

STEP 3 - APPLY SMOTE

SMOTE = Synthetic Minority Oversampling Technique
Right now, training data has 93.3% "no default" vs 6.7% "default". SMOTE looks at the existing defaulters, finds patterns between them, and creates new, synthetic defaulter examples — until both classes are roughly equal.
Critical rule: We apply SMOTE ONLY on X_train/y_train — NEVER on test data. The test set must remain real, untouched, original data — because that's what mimics real-world unseen data.

In [17]:
from imblearn.over_sampling import SMOTE
print('SMOTE imported successfully')

SMOTE imported successfully


In [18]:
smote = SMOTE(random_state = 42)   #create the SMOTE tool, reproducible results
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

#smote.fit_resample() -- look at training data, generate synthetic minority samples
#x_train_smote, y_train_smote -- new, balanced training data — these are what we'll train on

print('Before SMOTE:',X_train.shape,y_train.shape)
print('After SMOTE:',X_train_smote.shape,y_train_smote.shape)

print('\nclass distribution  after smote')
print(y_train_smote.value_counts(normalize=True) *100)


Before SMOTE: (119988, 10) (119988,)
After SMOTE: (223936, 10) (223936,)

class distribution  after smote
SeriousDlqin2yrs
0    50.0
1    50.0
Name: proportion, dtype: float64


Notice: X_train and y_train (real, original) still exist untouched — we only created NEW balanced versions for training. X_test/y_test are completely untouched.

INTERVIEW
"I applied SMOTE only on the training set after the train-test split, to avoid data leakage. It oversampled the minority class from 8,051 to roughly 112,000 records, balancing the training data to 50/50 — while the test set remained untouched at the original 93.3/6.7 ratio to reflect real-world conditions."

Data Leakage (easy definition):
Data leakage happens when the model accidentally gets information from the test data during training. This makes the model seem more accurate than it really is.

In your project:
If SMOTE is applied before splitting the data, some generated samples in the test set may be very similar to samples used for training. The model has indirectly "seen" the test data before.

Correct approach:
✅ Split the data into training and testing sets first.
✅ Apply SMOTE only to the training data.

This keeps the test data completely unseen and gives a realistic measure of model performance.

Interview answer (short):
"I applied SMOTE only after the train-test split to avoid data leakage. This ensures the model never sees information from the test set during training, giving a fair and reliable evaluation."

STEP 4 - TRAIN MODELS

from sklearn.linear_model import LogisticRegression
log_model = LogisticRegression(max_iter = 1000, random_state = 42)  
log_model.fit(X_train_smote,y_train_smote)
print('LogisticRegression trained successfully')

This warning means Logistic Regression could not finish learning properly within the allowed number of iterations.

1.max_iter = 1000 ---- Logistic Regression learns by repeatedly adjusting its weights to reduce error — this is an iterative process.
max_iter is the max number of adjustment rounds allowed. Default is 100, which sometimes isn't enough for the model to
fully converge (settle on stable weights), 
so we raise it to 1000 to be safe.
2.random_state = 42 ---- used to make random operations give the same result every time you run the code
3.(.fit(...) --- the model looks at the data and learns patterns
4.X_train_smote --- the 10 features (age, income, debt ratio, etc.) — the "questions"
5.y_train_smotethe answers (0 = no default, 1 = default)

Features on different scales mean some columns have much larger values than others. This can slow down Logistic Regression's optimization process, so we standardize the data to bring all features to a similar scale.

In [20]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)  #Looks at the training data and converts large and small values into a similar range.
X_test_scaled = scaler.transform(X_test)    #Applies the same conversion to the test data & no transform to avoid data leakage
print('scaling done')
print('\nsamples before scaling',X_train_smote.iloc[0].values)
print('\nsample after scaling',X_test_scaled[0])

scaling done

samples before scaling [9.999999e-01 2.600000e+01 0.000000e+00 0.000000e+00 1.600000e+03
 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 1.000000e+00]

sample after scaling [-0.02482716  1.00071496 -0.13645577 -0.19429549  0.01020688 -0.17594771
 -0.11389928  1.01022295 -0.10081261 -0.7725815 ]


In [21]:
log_model = LogisticRegression(max_iter = 1000, random_state = 42)
log_model.fit(X_train_scaled,y_train_smote)
print("Logistic Regression trained successfully on scaled data!")

Logistic Regression trained successfully on scaled data!


 STEP 5 - EVALUATING MODEL 1

In [22]:
from sklearn.metrics import classification_report,roc_auc_score
y_pred = log_model.predict(X_test_scaled)  #model predicts 0 or 1 for each test row
y_prob = log_model.predict_proba(X_test_scaled)[:,1]  #model gives probability of default (0 to 1) for each row 
#[:,1] --- take only the probability of class 1 (default) — we don't need class 0 probability
print('Classification report')
print(classification_report(y_test,y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob).round(4))   #compares real answers vs predicted probabilities

Classification report
              precision    recall  f1-score   support

           0       0.96      0.66      0.78     27993
           1       0.12      0.65      0.20      2005

    accuracy                           0.66     29998
   macro avg       0.54      0.66      0.49     29998
weighted avg       0.91      0.66      0.74     29998

ROC-AUC Score: 0.7044


Precision --- Of all people model predicted as defaulters, how many actually defaulted?
Recall --- Of all actual defaulters, how many did the model catch?
F1 Score --- Balance between precision and recall — single combined score
Support --- How many actual rows of each class exist in test set

For loan default — Recall is most important. Missing a real defaulter (giving them a loan they can't repay) costs the bank money. Better to be cautious

"Logistic Regression achieved a ROC-AUC of 0.70 and a default recall of 65%. While precision was low at 12%, this was expected given the complexity of financial data — Logistic Regression is a linear model and cannot capture non-linear relationships between features. This motivated us to try ensemble methods like Random Forest and XGBoost."

MODEL 2 - RANDOM FOREST CLASSIFIER

In [23]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(
    n_estimators = 100,   #build 100 decision trees
    random_state = 42,
    n_jobs = -1            #use ALL your CPU cores to train faster
)
rf_model.fit(X_train_smote,y_train_smote)  #notice we use unscaled data here — Random Forest doesn't need scaling unlike Logistic Regression
print("Random Forest trained successfully!")

Random Forest trained successfully!


Why no scaling needed for Random Forest?

Decision trees split data based on thresholds — "is income > 5000?" They don't care about the actual scale of numbers, just the order. So scaling makes no difference.

In [24]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:,1]
print('classification report:')
print(classification_report(y_test,y_pred_rf))
print('roc_auc_score:',roc_auc_score(y_test,y_prob_rf).round(4))


classification report:
              precision    recall  f1-score   support

           0       0.96      0.92      0.94     27993
           1       0.29      0.45      0.35      2005

    accuracy                           0.89     29998
   macro avg       0.62      0.68      0.64     29998
weighted avg       0.91      0.89      0.90     29998

roc_auc_score: 0.8242


Why Recall Dropped but Everything Else Improved
Logistic Regression was very aggressive — it flagged almost everyone as a defaulter to be safe. That's why it had high recall (catching more defaulters) but terrible precision (too many false alarms).
Random Forest is smarter and more conservative — it only flags someone as a defaulter when it's more confident. So it makes fewer false alarms (higher precision) but also misses more real defaulters (lower recall).
This is called the Precision-Recall tradeoff — improving one usually hurts the other.
For a bank — missing real defaulters is costly. So we want recall to be high. We'll see if XGBoost balances both better.

"Random Forest improved ROC-AUC from 0.70 to 0.82 and overall accuracy from 66% to 89%. The ensemble of 100 decision trees captured non-linear patterns that Logistic Regression missed. However recall for defaulters dropped to 0.45, showing a precision-recall tradeoff — which motivated us to try XGBoost for better balance."

MODEL 3 - XGBOOST 

In [25]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    n_estimators = 100, #build 100 sequential trees
    random_state = 42,
    n_jobs = -1,
    eval_metric = 'auc' #internally evaluate using ROC-AUC during training
)
xgb_model.fit(X_train_smote,y_train_smote)
print("XGBoost trained successfully!")
    

XGBoost trained successfully!


In [26]:
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:,1]
print('Classification report')
print(classification_report(y_test, y_pred_xgb))
print('roc-auc-score:',roc_auc_score(y_test, y_prob_xgb).round(4))
      

Classification report
              precision    recall  f1-score   support

           0       0.96      0.90      0.93     27993
           1       0.27      0.49      0.34      2005

    accuracy                           0.88     29998
   macro avg       0.61      0.70      0.64     29998
weighted avg       0.91      0.88      0.89     29998

roc-auc-score: 0.8205


Why didn't XGBoost clearly win?

This is actually very normal and important to understand. XGBoost doesn't always beat Random Forest — it depends on the dataset. Both are excellent ensemble models. The fact that they're neck and neck shows your data is well prepared.

Which Model Do You Choose?
For a bank — Random Forest is slightly better because:

Higher precision (0.29 vs 0.27) → fewer good customers wrongly rejected
Higher ROC-AUC (0.8242 vs 0.8205) → slightly better overall ranking
Higher accuracy (89% vs 88%)

But XGBoost has better recall (0.49 vs 0.45) → catches more defaulters
In practice a bank cares more about catching defaulters (recall) than false alarms — so this is a genuine tradeoff worth discussing in interviews.

Your Interview Answer

"I trained three models — Logistic Regression, Random Forest, and XGBoost. Logistic Regression achieved ROC-AUC of 0.70 but struggled with non-linear patterns. Both Random Forest and XGBoost achieved ROC-AUC of 0.82, significantly outperforming the baseline. I selected Random Forest as the final model due to its marginally higher precision and accuracy, though XGBoost showed better recall for defaulters — highlighting a precision-recall tradeoff relevant to real banking decisions.

In [27]:
results = {
    'Model' : ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy' : [0.66, 0.89, 0.88],
    'Default Recall' : [0.65, 0.45, 0.49],
    'Default Precision' : [0.12, 0.29, 0.27],
    'ROC-AUC' : [0.7044, 0.8242, 0.8205]
}
results_df = pd.DataFrame(results)
print(results_df)

                 Model  Accuracy  Default Recall  Default Precision  ROC-AUC
0  Logistic Regression      0.66            0.65               0.12   0.7044
1        Random Forest      0.89            0.45               0.29   0.8242
2              XGBoost      0.88            0.49               0.27   0.8205


In [28]:
import joblib   #library that saves Python objects to files
joblib.dump(rf_model, 'rf_model.pkl')   #saves the trained model to a file called rf_model.pkl
#.pkl stands for "pickle" — Python's format for saving objects
print("Model saved successfully!")

Model saved successfully!


Right now rf_model only exists in your notebook's memory. If you close Jupyter, it's gone. Saving it to a .pkl file means:

Your Streamlit app (Week 4) can load it directly
You don't need to retrain every time
Anyone can download your GitHub repo and use the model instantly

In [29]:
import os
print(os.getcwd())

C:\Users\Shweta\data


In [30]:
import joblib
joblib.dump(rf_model, "rf_model.pkl")
print("Model saved at:", os.getcwd() + "\\rf_model.pkl")

Model saved at: C:\Users\Shweta\data\rf_model.pkl


In [31]:
import joblib
import os

joblib.dump(rf_model, "rf_model.pkl")
print("Model saved successfully at:", os.getcwd())

Model saved successfully at: C:\Users\Shweta\data
